In [1]:
import pandas as pd 
import numpy as np 

In [2]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

## *Multiple-Choice Data Formatting*
*In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.*

*Q1. Label Encoding
Convert the answer column in train.csv into numeric labels using the following mapping:
A = 0
B = 1
C = 2
D = 3
E = 4
What is the encoded numeric label for the row at index 150?*

In [3]:
label_mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
train['encoded_answer'] = train['answer'].map(label_mapping)
encoded_label_150 = train.loc[150, 'encoded_answer']
print(encoded_label_150)

2


*Q2. Prompt-Option Formatting
For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)
What is the exact character length of this formatted input string?*


In [4]:
prompt = train.loc[0, 'prompt']
option_B = train.loc[0, 'B']

formatted_input = str(prompt) + " [SEP] " + str(option_B)
print(len(formatted_input))

407


## *Tokenization for Multiple-Choice Models*
*Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length*

*Since each question has five options, every row becomes five tokenized sequences.*

*Q3. Single-Row MCQ Tokenization
Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
padding = "max_length"
truncation = True
max_length = 128
return_tensors = "pt"*

*After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]*

*What is the value of the second dimension?*


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
options = ['A', 'B', 'C', 'D', 'E']
prompt = train.loc[0, 'prompt']

formatted_inputs = [str(prompt) + " [SEP] " + str(train.loc[0, opt]) for opt in options]

encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding['input_ids'].unsqueeze(0)
print(input_ids.shape[1])

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

5


*Q4. Batch MCQ Tokenization
Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.*

*The final input_ids tensor has shape:
[16, 5, 128]*

*How many total token positions are in this tensor?*

In [6]:
total_tokens = 16 * 5 * 128
print(total_tokens)

10240


## *Multiple-Choice Model Outputs*
*AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.*

*Q5. Multiple-Choice Logits
Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.*

*The output logits tensor has shape:
[1, 5]*

*How many logits are produced for one question?*

In [7]:
import torch
from transformers import AutoModelForMultipleChoice

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased").to(device)

input_ids = input_ids.to(device)

outputs = model(input_ids=input_ids)
logits = outputs.logits

print(logits.shape[1])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


5


*Q6. Supervised Loss Tensor
For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.*

*The model returns a scalar loss tensor.*

*How many dimensions does this loss tensor have?*


In [9]:
import torch

label = torch.tensor([train.loc[0, 'encoded_answer']]).to(device)

outputs = model(input_ids=input_ids, labels=label)
loss = outputs.loss

print(loss.ndim)

0


## *LoRA for Efficient Fine-Tuning*
*LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.*

*Q7. LoRA Trainable Parameters
Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS*

*Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)*

*How many parameters are trainable?*

In [11]:
!pip install -U torchao --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.7 MB/s eta 0:00:00:00:01


In [12]:
from peft import LoraConfig, get_peft_model, TaskType

# Define LoRA config
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    r=8, 
    lora_alpha=16, 
    target_modules=["query", "value"], 
    lora_dropout=0.1, 
    bias="none"
)

model = get_peft_model(model, peft_config)

print(sum(p.numel() for p in model.parameters() if p.requires_grad))

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


295681


Preparing Data for Hugging Face Trainer
Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.


Q8. Hugging Face Dataset Preparation
Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?

In [14]:
from datasets import Dataset

def tokenize_function(examples):
    first_sentences = []
    second_sentences = []
    
    options = ['A', 'B', 'C', 'D', 'E']
    
    for i in range(len(examples['prompt'])):
        prompt = examples['prompt'][i]
        for opt in options:
            first_sentences.append(str(prompt))
            second_sentences.append(str(examples[opt][i]))
            
    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        padding="max_length",
        truncation=True,
        max_length=128
    )
    
    num_rows = len(examples['prompt'])
    reshaped_input_ids = [tokenized['input_ids'][i*5:(i+1)*5] for i in range(num_rows)]
    reshaped_attention_mask = [tokenized['attention_mask'][i*5:(i+1)*5] for i in range(num_rows)]
    
    return {
        'input_ids': reshaped_input_ids,
        'attention_mask': reshaped_attention_mask,
        'label': examples['encoded_answer']
    }

train_dataset_100 = Dataset.from_pandas(train.iloc[:100]).map(tokenize_function, batched=True)

print(torch.tensor(train_dataset_100[0]['input_ids']).shape)  # Should output torch.Size([5, 128])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

torch.Size([5, 128])


Tiny Fine-Tuning and Inference
In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.
Q9. Tiny LoRA Fine-Tuning
Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

Q10. Probability Assigned to Option E After Fine-Tuning
Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places

In [13]:
options = ['A', 'B', 'C', 'D', 'E']
prompt = train.loc[0, 'prompt']
formatted_inputs = [str(prompt) + " [SEP] " + str(train.loc[0, opt]) for opt in options]

encoding = tokenizer(
    formatted_inputs, 
    padding="max_length", 
    truncation=True, 
    max_length=64, 
    return_tensors="pt"
)

input_ids = encoding['input_ids'].unsqueeze(0).to(device)
attention_mask = encoding['attention_mask'].unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # Shape: [1, 5]

probabilities = torch.softmax(logits, dim=-1).squeeze()

prob_E = probabilities[4].item()
print(f"{prob_E:.4f}")

0.2010
